# TDWI Lab 2: Local Agent — Wine Classifier

In this lesson you will co-write a spec with a **local Cursor agent**, plan and build a scikit-learn Wine classifier (tests + training code), run deterministic gates with **`scripts/check.sh`**, create the **`/commit-code`** slash command, and commit with sub-agent review.

See the **Lab map** in [README.md](README.md).

**Contrast with Lab 3:** Lab 2 stays **local** (inner loop at commit time). Lab 3 uses **Cursor Cloud Agents** and GitHub PR Automations for the outer loop.

## Learning Objectives

By the end of this mini-lesson you will be able to:
- Explore a minimal ML starter repo and understand what is provided vs what you will build
- Elicit requirements with a local agent and document them in `SPEC.md`
- Use **Plan mode** then **Agent mode** to write acceptance tests and implement a classifier
- Run deterministic gates with `bash scripts/check.sh` and `pytest`
- Create and use the **`/commit-code`** slash command for check → sub-agent review → commit message

## Prerequisites

- Completed [README.md](README.md) setup (fork, clone, local `.venv`, test push)
- `.venv` activated in Cursor's terminal (prompt shows `(.venv)`)
- Working on **`main`** of **your fork** in Cursor (e.g. `your-username/tdwi-sklearn-wine-starter`)
- **Local Agent** available in Cursor — this lab does **not** use Cloud Agents, Dockerfile, or secrets

**Important:** When prompting the agent, say **do not modify `README.md` or any lab notebook** — same guardrail as Lab 3.

## Step 1: Explore the starter repo

Open these files in Cursor and skim what exists:

- [`wine_data.py`](wine_data.py) — loads the Wine dataset and provides a fixed train/test split
- [`train_model.py`](train_model.py) — stub only; you will implement training in a later step
- [`SPEC.md`](SPEC.md) — empty project spec template (not [`requirements.txt`](requirements.txt)); you will fill this in with the agent

There is **no** `test_model.py` yet — you and the agent will create it from your spec. This is a **from-scratch build**: data plumbing is provided; the model and tests come from `SPEC.md`.

In a terminal at the project root with `.venv` active, run:

```bash
bash scripts/check.sh
```

This should **pass**. The script runs `pip install --dry-run` and `pytest .`. With no tests yet, pytest exits with code 5 (no tests collected) — `check.sh` treats that as OK.

Then run:

```bash
python train_model.py
```

You should see **`NotImplementedError`** — implementation is intentionally missing until you execute the plan.

## Step 2: Elicit requirements with the local agent

Open **Agent** chat in Cursor (sidebar **Agent** mode — **not** [cursor.com/agents](https://cursor.com/agents) Cloud Agents).

Paste this prompt:

```text
I want to build a wine cultivar classifier on the sklearn Wine dataset in this repo.
Before we implement anything, ask me clarifying questions about success metrics, modeling approach, deliverables, and what is out of scope.
Read wine_data.py and the empty SPEC.md template. Do not write code yet.
```

Go back and forth with the agent until you agree on scope. The agent should ask about things like:

- Which **metric**? (accuracy vs weighted F1)
- Use the **provided holdout** split in `wine_data.py` or redefine the split?
- **Where** to save the trained model?
- **Minimum** acceptable performance on the holdout set?
- What is **out of scope**? (deep learning, plots, REST API, etc.)

Your instructor may steer the class toward a bounded scope — for example logistic regression with `StandardScaler`, `GridSearchCV` over regularization `C` only, 5-fold CV on train, holdout accuracy ≥ 0.95, and saving to `models/wine_classifier.joblib`. Answer the agent's questions so your spec matches what the class agrees on.

### Optional: shortened path (instructor-led)

If time is tight, copy ideas from [`examples/specs/SPEC-example-bounded.md`](examples/specs/SPEC-example-bounded.md) into your `SPEC.md`, edit as needed, and skip to **Step 4**.

## Step 3: Co-write SPEC.md

When you and the agent agree on scope, ask it to write the spec. Paste:

```text
Based on our conversation, fill in SPEC.md using the template sections.
Include: test_model.py acceptance tests derived from success metrics, train_model.py implementation using wine_data.py, and models/wine_classifier.joblib as the artifact path.
Do not implement yet — only update SPEC.md.
```

1. Read the agent's draft in [`SPEC.md`](SPEC.md).
2. Edit anything that does not match what you agreed on.
3. Save the file.

You may commit `SPEC.md` alone now, or defer all commits to **Step 8** (recommended — one commit with code + command).

## Step 4: Plan from the spec (Plan mode)

Switch the chat to **Plan** mode (mode picker in Cursor chat).

Paste:

```text
Read SPEC.md and AGENTS.md. Create an implementation plan only — do not write or edit code yet.
The plan should cover: (1) test_model.py acceptance tests from the success metrics, (2) train_model.py implementation, (3) how we will verify with pytest and bash scripts/check.sh.
```

1. Read the plan.
2. Ask the agent to refine anything unclear.
3. When you approve the plan, switch back to **Agent** mode for **Step 5**.

This is **Recipe 2** in [WORKFLOW_RECIPES.md](WORKFLOW_RECIPES.md): [`AGENTS.md`](AGENTS.md) plus **pytest** define deterministic "done" — the agent runs tests and reacts to failures instead of you guessing whether the model is correct.

## Step 5: Write tests and implement the model

In **Agent** mode, execute the plan in two phases.

### Write acceptance tests first

```text
Implement the plan. First, create test_model.py with acceptance tests that match SPEC.md success metrics.
Use wine_data.py for data. Run python -m pytest and confirm tests fail (train_model.py is still a stub).
Do not modify README.md or any lab notebook.
```

Confirm `python -m pytest` reports **failures** — tests should fail until training code exists.

### Implement train_model.py

```text
Now implement train_model.py per SPEC.md. Loop on python -m pytest until all tests pass.
Also run python train_model.py and confirm holdout accuracy is printed.
```

Iterate until green:

1. Run `python -m pytest`.
2. Read failures with the agent.
3. Let the agent fix code and re-run tests.
4. Repeat until all tests pass and `python train_model.py` prints holdout accuracy.

## Step 6: Run deterministic checks (Recipe 1)

From the project root with `.venv` active:

```bash
bash scripts/check.sh
```

This script runs:

1. `pip install --dry-run -r requirements.txt` — catches dependency conflicts before install
2. `python -m pytest .` — runs all acceptance tests in the repo

Fix any failures and re-run until exit code **0**. Do not continue to **Step 8** until `check.sh` passes.

This is **Recipe 1** in [WORKFLOW_RECIPES.md](WORKFLOW_RECIPES.md). **`scripts/check.sh`** was **pre-shipped** on this lab — unlike Lab 3 Part 3 Step 8, where students create `check.sh` in class.

## Step 7: Create the `/commit-code` slash command (Recipe 3)

Slash commands in Cursor are **reusable prompts**: a file under `.cursor/commands/` becomes `/command-name` in chat. Here you create the local commit workflow — the same **check → sub-agent review → commit message** pattern Lab 3 wires into a Cloud Agent **golden prompt** before opening a PR.

**Important:** Create this command only after **Step 5** — you need a meaningful code diff to commit.

1. In your repo root, create the folder `.cursor/commands/` if it does not exist.
2. Create the file `.cursor/commands/commit-code.md`.
3. Paste the contents below (you can also copy from [`examples/cursor/commands/commit-code.md`](examples/cursor/commands/commit-code.md), omitting the "EXAMPLE ONLY" header):

```markdown
# commit-code

Optional arguments: `--quick` or `--no-review` (skip the expensive sub-agent review step).

---

Review the current git diff. Then run deterministic checks before suggesting a commit:

1. Run `bash scripts/check.sh` from the repo root.
2. If checks fail, stop and summarize failures—do not suggest a commit.
3. Unless the user passed `--quick` or `--no-review`, launch a **fresh sub-agent** (or separate review pass) to review only the diff. The reviewer must not be the same context that wrote the changes. Focus on scope, correctness risks, and test coverage—not re-running the full implementation.
   - **Note:** Sub-agent review is **slow and costly** (extra model calls). Use it as the last automated gate before a human commit/merge, or skip with `--quick` while your team is still adopting the workflow.
4. If review or checks surface issues, summarize them for the user.
5. If all gates are green, draft a concise commit message from the **staged** diff (`git diff --cached`) and ask the user whether to commit.

Do not commit unless the user explicitly confirms.
```

4. Save the file. In Agent chat you should now be able to type **`/commit-code`**.

This is **Recipe 3** in [WORKFLOW_RECIPES.md](WORKFLOW_RECIPES.md).

## Step 8: Stage changes and run `/commit-code`

1. Open the **Source Control** tab in Cursor.
2. Stage these files (click **+** next to each):
   - `SPEC.md`
   - `test_model.py`
   - `train_model.py`
   - `.cursor/commands/commit-code.md`
   - Do **not** stage `models/wine_classifier.joblib` — model artifacts are gitignored and stay local.
3. In **Agent** chat, run **`/commit-code`** (or **`/commit-code --quick`** if your instructor demos skipping sub-agent review).
4. Watch the workflow:
   - **`check.sh`** must pass first
   - Unless you used `--quick`, a **fresh sub-agent** reviews the **staged** diff
   - The agent proposes a commit message from `git diff --cached`
5. Read the review and proposed message. **Confirm the commit only if you agree** — the command does not commit without your approval.

| Gate | What it checks |
|------|----------------|
| `check.sh` | Cheap deterministic failures (deps, tests) |
| Sub-agent review | Scope, risks, coverage — expensive; skip with `--quick` while adopting |
| You | Final commit decision |

**Lab 3 contrast:** Part 3 Step 9 embeds the same inner-loop pattern in a **Cloud Agent golden prompt** before opening a PR. Lab 2 embeds it in a **local slash command** at commit time. Part 3 then adds PR Automations as the **outer** review loop.

## Key Takeaways

- **Spec-first:** `SPEC.md` before code — the agent asks clarifying questions instead of guessing scope.
- **Tests from the spec:** acceptance tests encode "done"; the agent implements until `pytest` is green.
- **Escalating cost:** `check.sh` (cheap) → sub-agent review (expensive) → human commit. Use `/commit-code --quick` while your team is still adopting.
- **Harness split:** Lab 2 ships **`check.sh`** and creates **`/commit-code`** in class. Lab 3 creates **`check.sh`** in class and keeps `/commit-code` in `examples/` only.
- **Recipes 1–3** in [WORKFLOW_RECIPES.md](WORKFLOW_RECIPES.md) are the adoption path for local workflows; Labs 4–6 in that guide are optional post-lab extensions.

## Debrief questions

1. What did `SPEC.md` give the agent that a one-line "build a classifier" prompt would not?
2. Why write acceptance tests from the spec before (or alongside) implementation?
3. What does `/commit-code` automate vs what must you still decide as the human?
4. How does this local inner loop relate to Lab 3's Cloud Agent golden prompt and PR Automations?

Lab 2 is complete. If your workshop includes Lab 3, continue with the Lab 3 notebooks in the sales pipeline repo for Cloud Agents and the outer review loop.